# RAG Pipeline — ร้านฟ้าใหม่ (FahMai)
In-Memory Retrieval-Augmented Generation for Multiple Choice QA

---
**Pipeline Overview:**
1. Ingest `.md` knowledge base files into RAM
2. Vectorize documents with `BAAI/bge-m3`
3. Load `questions.csv` (id, question, choice_1…choice_10)
4. Cosine-similarity retrieval + trace logging
5. LLM generation via Ollama API
6. Fill `sample_submission.csv` template (id, ans) and save to `submission/`

## Cell 1 — Configuration Constants

In [7]:
# ============================================================
#  Configuration Constants
# ============================================================

KNOWLEDGE_BASE_DIR     = "../data/knowledge_base"
QUESTIONS_CSV_PATH     = "../data/questions.csv"
SAMPLE_SUBMISSION_PATH = "../data/sample_submission.csv"
SUBMISSION_DIR         = "../submission"
TRACE_LOG_PATH         = "./trace/trace_log.jsonl"

OLLAMA_API_URL         = "http://localhost:11434/api/generate"
LLM_MODEL_NAME         = "hf.co/nectec/pathumma-thaillm-8b-think-3.0.0-GGUF:Q8_0"
EMBEDDING_MODEL_NAME   = "Qwen/Qwen3-Embedding-0.6B"
TOP_K_RETRIEVAL        = 3

print("✅ Constants loaded.")

✅ Constants loaded.


## Cell 2 — Imports & Directory Setup

In [8]:
import os
import json
import re
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_text_splitters import RecursiveCharacterTextSplitter

Path(TRACE_LOG_PATH).parent.mkdir(parents=True, exist_ok=True)
Path(SUBMISSION_DIR).mkdir(parents=True, exist_ok=True)

print("✅ Imports done. Directories ready.")

✅ Imports done. Directories ready.


## Cell 3 — Step 1: In-Memory Data Ingestion

In [9]:
# ============================================================
#  Step 1 — Load and CHUNK .md files
# ============================================================

# กำหนดขนาด Chunk (เช่น 500 ตัวอักษร, เผื่อส่วนที่ซ้อนทับกัน 50 ตัวอักษร)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=100,
    separators=["\n\n## ", "\n\n", "\n", " "]
)

documents = []

for root, dirs, files in os.walk(KNOWLEDGE_BASE_DIR):
    for fname in sorted(files):
        if fname.endswith(".md"):
            filepath = os.path.join(root, fname)
            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    content = f.read().strip()
                
                # หั่นเนื้อหาออกเป็นชิ้นเล็กๆ
                chunks = text_splitter.split_text(content)
                
                # เก็บแต่ละ Chunk แยกเป็น 1 Document
                for i, chunk in enumerate(chunks):
                    documents.append({
                        "filename": f"{os.path.relpath(filepath, KNOWLEDGE_BASE_DIR)} (Chunk {i+1})",
                        "content": chunk
                    })
            except Exception as e:
                print(f"⚠️  Could not read {filepath}: {e}")

print(f"✅ Loaded {len(documents)} chunks into memory.")

✅ Loaded 1337 chunks into memory.


## Cell 4 — Step 2: Vectorization (No Database)

In [10]:
# ============================================================
#  Step 2 — Embed all documents into a numpy matrix in RAM
# ============================================================

print(f"🔄 Loading embedding model: {EMBEDDING_MODEL_NAME} ...")
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

doc_texts = [f"Source: {doc['filename']}\n{doc['content']}" for doc in documents]

print(f"🔄 Encoding {len(doc_texts)} chunks...")
doc_embeddings = embed_model.encode(
    doc_texts,
    batch_size=16, # หาก RAM/VRAM เยอะ สามารถปรับเป็น 32 หรือ 64 เพื่อให้เร็วขึ้นได้
    show_progress_bar=True,
    normalize_embeddings=True  # unit-norm → cosine sim == dot product
)

print(f"✅ Document matrix shape: {doc_embeddings.shape}")

🔄 Loading embedding model: Qwen/Qwen3-Embedding-0.6B ...


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 30451.16it/s]
'The read operation timed out' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main/special_tokens_map.json
Retrying in 1s [Retry 1/5].


🔄 Encoding 1337 chunks...


Batches: 100%|██████████| 84/84 [22:25<00:00, 16.02s/it]

✅ Document matrix shape: (1337, 1024)


## Cell 5 — Step 3: Load Questions & Submission Template

In [11]:
# ============================================================
#  Step 3 — Load questions.csv and sample_submission.csv
# ============================================================

# questions.csv: [id, question, choice_1 ... choice_10]
df_questions = pd.read_csv(QUESTIONS_CSV_PATH)
CHOICE_COLS  = [f"choice_{i}" for i in range(1, 11)]

missing_q = [c for c in ["id", "question"] + CHOICE_COLS if c not in df_questions.columns]
assert not missing_q, f"Missing columns in questions.csv: {missing_q}"
print(f"✅ Questions loaded: {len(df_questions)} rows")
display(df_questions.head(3))

# sample_submission.csv: [id, answer] — used as the output template
df_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
assert "id" in df_submission.columns and "answer" in df_submission.columns, \
    "sample_submission.csv must have columns: id, answer"
print(f"\n✅ Sample submission template loaded: {len(df_submission)} rows")
display(df_submission.head(3))

✅ Questions loaded: 100 rows


,id,question,choice_1,choice_2,choice_3,choice_4,choice_5,choice_6,choice_7,choice_8,choice_9,choice_10
0,1,Watch S3 Ultra กันน้ำได้กี่ ATM ครับ,3 ATM,IP68,5 ATM,IP67,10 ATM,20 ATM,IPX8,1 ATM,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
1,2,ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ,"2,990 บาท","4,490 บาท","1,990 บาท","4,990 บาท","3,490 บาท","2,490 บาท","3,990 บาท","5,490 บาท",ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
2,3,หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ,Bluetooth 5.0,Bluetooth 5.3,Bluetooth 5.4,Bluetooth 4.2,Bluetooth 5.2,Bluetooth 5.1,Bluetooth 4.0,Bluetooth 5.5,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า



✅ Sample submission template loaded: 100 rows


,id,answer
0,1,1
1,2,1
2,3,1


In [12]:
df_questions.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   id         100 non-null    int64
 1   question   100 non-null    str  
 2   choice_1   100 non-null    str  
 3   choice_2   100 non-null    str  
 4   choice_3   100 non-null    str  
 5   choice_4   100 non-null    str  
 6   choice_5   100 non-null    str  
 7   choice_6   100 non-null    str  
 8   choice_7   100 non-null    str  
 9   choice_8   100 non-null    str  
 10  choice_9   100 non-null    str  
 11  choice_10  100 non-null    str  
dtypes: int64(1), str(11)
memory usage: 9.5 KB


## Cell 6 — Helper Functions

In [16]:
# ============================================================
#  Helper Functions (Upgraded Version 🚀)
# ============================================================

def retrieve_top_k(question: str, max_k: int = 4, threshold: float = 0.45) -> list[dict]:
    """
    Dynamic Retrieval: Embeds the question and returns documents 
    that pass the similarity threshold, up to a maximum of max_k.
    """
    q_vec  = embed_model.encode([question], normalize_embeddings=True)
    scores = cosine_similarity(q_vec, doc_embeddings)[0]  # shape (n_docs,)
    
    # Sort from highest score to lowest, taking the top max_k
    top_indices = np.argsort(scores)[::-1][:max_k]
    
    results = []
    for i in top_indices:
        score = float(scores[i])
        
        # DYNAMIC LOGIC: Only keep the document if its score is above the threshold
        if score >= threshold:
            results.append({
                "filename": documents[i]["filename"],
                "content":  documents[i]["content"],
                "score":    score
            })
            
    # SAFETY NET: What if the question is tricky and ALL scores are below the threshold?
    # We still want to give the LLM the absolute closest match to try and figure it out.
    if len(results) == 0:
        best_idx = top_indices[0]
        results.append({
            "filename": documents[best_idx]["filename"],
            "content":  documents[best_idx]["content"],
            "score":    float(scores[best_idx])
        })
        
    return results

def build_prompt(question: str, choices: list[str], retrieved_docs: list[dict]) -> str:
    """Build a highly detailed and strictly constrained Thai RAG prompt."""
    
    context = "\n\n---\n\n".join(
        f"[{doc['filename']}]\n{doc['content']}" for doc in retrieved_docs
    )
    choice_lines = "\n".join(f"{i+1}. {c}" for i, c in enumerate(choices))
    
    return f"""คุณคือ AI ผู้ช่วยอัจฉริยะระดับสูงของร้านฟ้าใหม่ (FahMai) ผู้ทำหน้าที่ผู้ประเมินข้อสอบแบบปรนัย (Multiple Choice)
    คำสั่งสูงสุดของคุณคือ: "ต้องตอบคำถามโดยอ้างอิงหลักฐานจาก [เอกสารอ้างอิง] ที่แนบมาให้เท่านั้น ห้ามใช้ความรู้เดิม ห้ามคิดไปเอง และห้ามเดาเด็ดขาด"

    [เอกสารอ้างอิง]
    {context}

    [คำถาม]
    {question}

    [ตัวเลือก]
    {choice_lines}

    [กฎการวิเคราะห์และข้อกำหนดที่ต้องปฏิบัติตามอย่างเคร่งครัด]
    1. ขั้นตอนการวิเคราะห์ (ภายในแท็ก <think>):
    - อ่านคำถามและดึงคีย์เวิร์ดสำคัญ
    - ค้นหาคีย์เวิร์ดเหล่านั้นใน [เอกสารอ้างอิง] แบบคำต่อคำ
    - หากพบข้อมูล: ให้นำข้อมูลนั้นมาเทียบกับ [ตัวเลือก] เพื่อหาข้อที่มีความหมายตรงกันที่สุดเพียง 1 ข้อ
    - หากไม่พบข้อมูล: หากอ่านเอกสารทั้งหมดแล้วไม่มีคำตอบ ห้ามพยายามเดา ให้เลือกตัวเลือกที่มีความหมายว่า "ไม่มีข้อมูลนี้ในฐานข้อมูล" ทันที

    2. การจัดการความยาว (ภายในแท็ก <think>):
    - จงเขียนสรุปการคิดให้ "สั้น กระชับ และตรงประเด็นที่สุด" (เช่น "ไฟล์ X ระบุว่าราคา 1,990 บาท ตรงกับตัวเลือก 3")
    - ห้ามเขียนเรียงความยาวๆ เพื่อป้องกันปัญหา Token Limit ทะลุ

    3. กฎการตอบผลลัพธ์สุดท้าย (ภายนอกแท็ก <think>):
    - บรรทัดสุดท้ายของคำตอบ จะต้องเป็นคำว่า "คำตอบ: " ตามด้วย "เลขโดด (1-10)" เท่านั้น
    - ห้ามใส่สัญลักษณ์ ห้ามใส่วงเล็บ ห้ามทวนคำถาม และห้ามพิมพ์ข้อความใดๆ ต่อท้ายตัวเลขโดยเด็ดขาด

    [รูปแบบผลลัพธ์ที่คุณต้องสร้าง (บังคับ)]
    <think>""" 

def call_ollama(prompt: str) -> str:
    """POST prompt to Ollama and return the raw response text."""
    payload = {
        "model": LLM_MODEL_NAME, 
        "prompt": prompt, 
        "stream": False,
        "options": {
            "num_ctx": 8192,
            "num_predict": 2048,
            "temperature": 0.1,
            "top_p": 0.9,
            "repeat_penalty": 1.15
        }
    }
    try:
        resp = requests.post(OLLAMA_API_URL, json=payload, timeout=600)
        resp.raise_for_status()
        
        raw = resp.json().get("response", "")
        
        # เนื่องจากเราบังคับเปิดแท็ก <think> ไปใน Prompt แล้ว
        # ผลลัพธ์ที่ได้กลับมามักจะไม่มี <think> เปิดแล้ว เราจึงประกอบร่างให้สมบูรณ์เพื่อง่ายต่อการ parse
        return "<think>\n" + raw 

    except Exception as e:
        print(f"  ⚠️ Ollama API Error: {e}")
        return ""

def extract_think(raw: str) -> str:
    """Extract content inside <think>...</think> tags."""
    if not raw: return ""
    m = re.search(r"<think>(.*?)</think>", raw, re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else ""

def extract_final_answer(raw: str) -> str:
    """Extract the text after </think> or after the word 'คำตอบ:'."""
    if not raw: return ""
    
    # 1. ตัด <think>...</think> ทิ้ง
    clean = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL | re.IGNORECASE).strip()
    
    # 2. ค้นหาคำว่า "คำตอบ:" แล้วดึง *เฉพาะตัวเลข* ชุดแรกที่เจอ
    m = re.search(r"คำตอบ\s*:\s*.*?(\d+)", clean)
    if m:
        return m.group(1).strip()
        
    # 3. Fallback: ถ้าไม่มีแพทเทิร์น ให้พยายามหาตัวเลขในบรรทัดสุดท้าย
    lines = [l.strip() for l in clean.splitlines() if l.strip()]
    if lines:
        last_line = lines[-1]
        digit_match = re.search(r"(\d+)", last_line)
        if digit_match:
            return digit_match.group(1)
        return last_line
    return clean

def append_trace(record: dict):
    """Append one JSON record to the trace log (JSONL format)."""
    with open(TRACE_LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def clear_trace():
    """Clear the trace log by truncating the file (JSONL format)."""
    if os.path.exists(TRACE_LOG_PATH):
        with open(TRACE_LOG_PATH, "w", encoding="utf-8"):
            pass

print("✅ Helper functions upgraded.")

✅ Helper functions upgraded.


## Cell 7 — Steps 4–6: Main Pipeline Loop

In [17]:
# ============================================================
#  Steps 4-6 — Retrieval → LLM → Parse → Fill submission
# ============================================================

# Work on a copy of the template; iterate by id order from sample_submission
df_result = df_submission.copy()
df_result["answer"] = df_result["answer"].astype(object) 
q_lookup = df_questions.set_index("id")

clear_trace()

for idx, row in df_result.iterrows():
    q_id = int(row["id"])

    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(df_result)}] ID={q_id}")

    try:
        q_row    = q_lookup.loc[q_id]
        question = str(q_row["question"])
        choices  = [str(q_row[c]) for c in CHOICE_COLS]

        print(f"  ❓ {question[:80]}{'...' if len(question) > 80 else ''}")

        # --- Step 4: Retrieve ---
        retrieved           = retrieve_top_k(question)
        retrieved_filenames = [r["filename"] for r in retrieved]
        print(f"  📄 Retrieved: {retrieved_filenames}")

        # --- Step 5: LLM Generation ---
        prompt   = build_prompt(question, choices, retrieved)
        raw_resp = call_ollama(prompt)

        # --- Step 6: Parse ---
        think_text = extract_think(raw_resp)
        raw_answer = extract_final_answer(raw_resp)

        match_index = None
        is_fallback = False
        clean_raw = str(raw_answer).strip()

        # หาตัวเลข 1-10 จากผลลัพธ์ที่ extract มาได้
        numbers = re.findall(r'\b([1-9]|10)\b', clean_raw) 
        if numbers:
            match_index = int(numbers[0])
        else:
            match_index = 1 # Absolute fallback
            is_fallback = True

        # --- Explicit Print Logging ---
        if is_fallback:
            print(f"  ⚠️ FALLBACK (Defaulting to 1). Raw: '{str(raw_answer).strip()[:40]}...'")
        else:
            print(f"  🤖 Extracted Answer: {match_index}")

        # ---> รวบ append_trace ไว้ที่เดียว (1 ข้อ = 1 บรรทัดใน Log)
        append_trace({
            "question_id":     q_id,
            "question":        question,
            "retrieved_files": retrieved_filenames,
            "raw_response":    raw_resp,
            "think":           think_text,
            "final_answer":    match_index,
            "is_fallback":     is_fallback 
        })

        # Write the NUMBER into the submission DataFrame
        df_result.at[idx, "answer"] = int(match_index)

    except Exception as e:
        err_msg = str(e)
        print(f"  ❌ ERROR: {err_msg}")
        
        # กรณีเกิด Error ก็ Log แบบรวบยอดเช่นกัน
        append_trace({
            "question_id": q_id, 
            "error": err_msg,
            "is_fallback": True
        })

# --- Save timestamped submission CSV ---
timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(SUBMISSION_DIR, f"submission_{timestamp}.csv")
df_result.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"\n✅ Done! Submission saved → {output_path}")
display(df_result)


[1/100] ID=1
  ❓ Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
  📄 Retrieved: ['products/WK-SW-001_wongkhojon_watch_s3_ultra.md (Chunk 3)', 'products/WK-SW-001_wongkhojon_watch_s3_ultra.md (Chunk 15)', 'products/WK-SW-001_wongkhojon_watch_s3_ultra.md (Chunk 16)', 'products/WK-SW-002_wongkhojon_watch_s3_pro.md (Chunk 16)']
  🤖 Extracted Answer: 5

[2/100] ID=2
  ❓ ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ
  📄 Retrieved: ['products/SF-TB-001_saifah_tab_s9_pro.md (Chunk 16)', 'products/JC-CS-006_judchuam_saifah_pen_gen2.md (Chunk 1)', 'products/SF-TB-005_saifah_tab_mini_7.md (Chunk 12)', 'products/JC-CS-006_judchuam_saifah_pen_gen2.md (Chunk 2)']
  🤖 Extracted Answer: 7

[3/100] ID=3
  ❓ หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ
  📄 Retrieved: ['products/KS-HP-001_kluensiang_headpro_x1.md (Chunk 10)', 'products/KS-HP-001_kluensiang_headpro_x1.md (Chunk 12)', 'products/KS-HP-002_kluensiang_headpro_x1_se.md (Chunk 2)', 'products/KS-HP-001_kluensiang_headpro_x1.md (Chunk 2)']
  🤖 Extracted Answer: 2

,id,answer
0,1,5
1,2,7
2,3,2
3,4,6
4,5,6
...,...,...
95,96,9
96,97,1
97,98,1
98,99,9


## Cell 8 — Sanity Check

In [18]:
# ============================================================
#  Sanity Check — submission stats & trace entries
# ============================================================

print(f"Total rows         : {len(df_result)}")
print(f"Answered (non-null): {df_result['answer'].notna().sum()}")
print(f"Still template val : {(df_result['answer'] == df_submission['answer']).sum()}")
print()
display(df_result)

with open(TRACE_LOG_PATH, "r", encoding="utf-8") as f:
    trace_lines = f.readlines()
print(f"\nTrace log entries: {len(trace_lines)}")

Total rows         : 100
Answered (non-null): 100
Still template val : 17



,id,answer
0,1,5
1,2,7
2,3,2
3,4,6
4,5,6
...,...,...
95,96,9
96,97,1
97,98,1
98,99,9



Trace log entries: 100
